In [1]:
import subprocess, sys as _sys_tmp
for _w in [
    '/kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl',
    '/kaggle/input/datasets/amirrezaaleyasin/biotite/biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl',
    '/kaggle/input/datasets/amirrezaaleyasin/rdkit-2025-9-5/rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl',
    '/kaggle/input/ml-collections/ml_collections-1.0.0-py3-none-any.whl',
]:
    try:
        subprocess.check_call([_sys_tmp.executable, '-m', 'pip', 'install', '-q', '--no-deps', _w])
        print(f"  OK: {_w.split('/')[-1]}")
    except Exception:
        print(f"  SKIP: {_w.split('/')[-1]}")

  OK: biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
  OK: biotite-1.6.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
  OK: rdkit-2025.9.5-cp312-cp312-manylinux_2_28_x86_64.whl
  OK: ml_collections-1.0.0-py3-none-any.whl


In [2]:
import gc, json, os, sys, time
from pathlib import Path
os.environ["LAYERNORM_TYPE"] = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")
import numpy as np
import pandas as pd
import torch
from Bio.Align import PairwiseAligner
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

IS_KAGGLE = True
LOCAL_N_SAMPLES = None

DATA_BASE              = "/kaggle/input/stanford-rna-3d-folding-2"
DEFAULT_TEST_CSV       = f"{DATA_BASE}/test_sequences.csv"
DEFAULT_TRAIN_CSV      = f"{DATA_BASE}/train_sequences.csv"
DEFAULT_TRAIN_LBLS     = f"{DATA_BASE}/train_labels.csv"
DEFAULT_VAL_CSV        = f"{DATA_BASE}/validation_sequences.csv"
DEFAULT_VAL_LBLS       = f"{DATA_BASE}/validation_labels.csv"
DEFAULT_OUTPUT         = "/kaggle/working/submission.csv"

DEFAULT_CODE_DIR = (
    "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted"
    "/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
)
DEFAULT_ROOT_DIR = DEFAULT_CODE_DIR

MODEL_NAME    = "protenix_base_20250630_v1.0.0"
N_SAMPLE      = 5
SEED          = 42
MAX_SEQ_LEN   = int(os.environ.get("MAX_SEQ_LEN",   "512"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP",  "128"))

MIN_SIMILARITY       = float(os.environ.get("MIN_SIMILARITY",       "0.0"))
MIN_PERCENT_IDENTITY = float(os.environ.get("MIN_PERCENT_IDENTITY", "50.0"))

USE_PROTENIX = True

def parse_bool(value, default=False):
    v = str(value).strip().lower()
    if v in {"1","true","t","yes","y","on"}: return "true"
    if v in {"0","false","f","no","n","off"}: return "false"
    return "true" if default else "false"

USE_MSA      = parse_bool(os.environ.get("USE_MSA",      "false"))
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA  = parse_bool(os.environ.get("USE_RNA_MSA",  "true"))
MODEL_N_SAMPLE = int(os.environ.get("MODEL_N_SAMPLE", str(N_SAMPLE)))

In [3]:
def seed_everything(seed):
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)

def resolve_paths():
    test_csv   = os.environ.get("TEST_CSV",          DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV",    DEFAULT_OUTPUT)
    code_dir   = os.environ.get("PROTENIX_CODE_DIR", DEFAULT_CODE_DIR)
    root_dir   = os.environ.get("PROTENIX_ROOT_DIR", DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir

def ensure_required_files(root_dir):
    for p, name in [
        (Path(root_dir)/"checkpoint"/f"{MODEL_NAME}.pt",         "checkpoint"),
        (Path(root_dir)/"common"/"components.cif",               "CCD file"),
        (Path(root_dir)/"common"/"components.cif.rdkit_mol.pkl", "CCD cache"),
    ]:
        if not p.exists():
            raise FileNotFoundError(f"Missing {name}: {p}")

def build_input_json(df, json_path):
    data = [{"name": row["target_id"], "covalent_bonds": [],
             "sequences": [{"rnaSequence": {"sequence": row["sequence"], "count": 1}}]}
            for _, row in df.iterrows()]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)

def build_configs(input_json_path, dump_dir, model_name):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs
    base = {**configs_base, **{"data": data_configs}, **inference_configs}
    def deep_update(t, p):
        for k, v in p.items():
            if isinstance(v, dict) and k in t and isinstance(t[k], dict): deep_update(t[k], v)
            else: t[k] = v
    deep_update(base, model_configs[model_name])
    arg_str = " ".join([
        f"--model_name {model_name}",
        f"--input_json_path {input_json_path}",
        f"--dump_dir {dump_dir}",
        f"--use_msa {USE_MSA}",
        f"--use_template {USE_TEMPLATE}",
        f"--use_rna_msa {USE_RNA_MSA}",
        f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
        f"--seeds {SEED}",
    ])
    return parse_configs(configs=base, arg_str=arg_str, fill_required_with_null=True)

def get_c1_mask(data, atom_array):
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                m = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"): m = m & atom_array.is_rna
                return torch.from_numpy(m).bool()
            if hasattr(atom_array, "atom_name"):
                base = atom_array.atom_name == "C1'"
                if hasattr(atom_array, "is_rna"): base = base & atom_array.is_rna
                return torch.from_numpy(base).bool()
        except: pass
    f = data["input_feature_dict"]
    if "centre_atom_mask" in f: return (f["centre_atom_mask"] == 1).bool()
    if "center_atom_mask" in f: return (f["center_atom_mask"] == 1).bool()
    n_tokens = data.get("N_token", torch.tensor(0)).item()
    m11 = (f["atom_to_tokatom_idx"] == 11).bool()
    m12 = (f["atom_to_tokatom_idx"] == 12).bool()
    return m11 if abs(m11.sum().item()-n_tokens) < abs(m12.sum().item()-n_tokens) else m12

def coords_to_rows(target_id, seq, coords):
    rows = []
    for i in range(len(seq)):
        row = {"ID": f"{target_id}_{i+1}", "resname": seq[i], "resid": i+1}
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]: x,y,z = coords[s,i]
            else: x,y,z = 0.0,0.0,0.0
            row[f"x_{s+1}"] = float(x); row[f"y_{s+1}"] = float(y); row[f"z_{s+1}"] = float(z)
        rows.append(row)
    return rows

def split_into_chunks(seq_len, max_len, overlap):
    if seq_len <= max_len: return [(0, seq_len)]
    chunks, step, pos = [], max_len-overlap, 0
    while pos < seq_len:
        end = min(pos+max_len, seq_len); chunks.append((pos, end))
        if end == seq_len: break
        pos += step
    return chunks

def kabsch_align(P, Q):
    cP, cQ = P.mean(0), Q.mean(0); Pc, Qc = P-cP, Q-cQ
    H = Pc.T @ Qc; U, _, Vt = np.linalg.svd(H)
    d = np.linalg.det(Vt.T @ U.T); S = np.eye(3)
    if d < 0: S[2,2] = -1
    R = Vt.T @ S @ U.T
    return R, cQ - R @ cP

def stitch_chunk_coords(chunk_coords_list, chunk_ranges, seq_len):
    if len(chunk_coords_list) == 1:
        coords = chunk_coords_list[0]
        if coords.shape[0] >= seq_len: return coords[:seq_len]
        out = np.zeros((seq_len,3), dtype=coords.dtype); out[:coords.shape[0]] = coords; return out
    aligned = [chunk_coords_list[0].copy()]
    for i in range(1, len(chunk_coords_list)):
        ps,pe = chunk_ranges[i-1]; cs,ce = chunk_ranges[i]
        ov_s,ov_e = cs, min(pe,ce)
        if ov_e-ov_s < 3: aligned.append(chunk_coords_list[i].copy()); continue
        prev_ov = aligned[i-1][ov_s-ps:ov_e-ps]; cur_ov = chunk_coords_list[i][ov_s-cs:ov_e-cs]
        valid = ~(np.isnan(prev_ov).any(1)|np.isnan(cur_ov).any(1))
        if valid.sum() < 3: aligned.append(chunk_coords_list[i].copy()); continue
        R,t = kabsch_align(cur_ov[valid], prev_ov[valid])
        aligned.append((chunk_coords_list[i] @ R.T) + t)
    full = np.zeros((seq_len,3), dtype=np.float64); weights = np.zeros(seq_len, dtype=np.float64)
    for i, ((s,e), coords) in enumerate(zip(chunk_ranges, aligned)):
        cl = coords.shape[0]; ae = min(s+cl,seq_len); ul = ae-s
        w = np.ones(ul, dtype=np.float64)
        if i > 0:
            ov_e2 = min(chunk_ranges[i-1][1],e); rl = ov_e2-s
            if rl > 0: w[:rl] = np.linspace(0.,1.,rl)
        if i < len(chunk_ranges)-1:
            ns2 = chunk_ranges[i+1][0]; rs = ns2-s; rl = ae-ns2
            if rl > 0 and rs < ul: w[rs:ul] = np.linspace(1.,0.,rl)
        full[s:ae] += coords[:ul]*w[:,None]; weights[s:ae] += w
    mask = weights > 0; full[mask] /= weights[mask,None]
    return full

def _make_aligner():
    al = PairwiseAligner()
    al.mode = "global"; al.match_score = 2; al.mismatch_score = -1.5
    al.open_gap_score = -8; al.extend_gap_score = -0.4
    al.query_left_open_gap_score = -8;   al.query_left_extend_gap_score = -0.4
    al.query_right_open_gap_score = -8;  al.query_right_extend_gap_score = -0.4
    al.target_left_open_gap_score = -8;  al.target_left_extend_gap_score = -0.4
    al.target_right_open_gap_score = -8; al.target_right_extend_gap_score = -0.4
    return al

_aligner = _make_aligner()

def parse_stoichiometry(stoich):
    if pd.isna(stoich) or str(stoich).strip() == "": return []
    return [(ch.strip(), int(cnt)) for part in str(stoich).split(";") for ch,cnt in [part.split(":")]]

def parse_fasta(fasta_content):
    out, cur, parts = {}, None, []
    for line in str(fasta_content).splitlines():
        line = line.strip()
        if not line: continue
        if line.startswith(">"):
            if cur is not None: out[cur] = "".join(parts)
            cur = line[1:].split()[0]; parts = []
        else: parts.append(line.replace(" ",""))
    if cur is not None: out[cur] = "".join(parts)
    return out

def get_chain_segments(row):
    seq = row["sequence"]; stoich = row.get("stoichiometry",""); all_sq = row.get("all_sequences","")
    if pd.isna(stoich) or pd.isna(all_sq) or str(stoich).strip()==""or str(all_sq).strip()=="":
        return [(0,len(seq))]
    try:
        cd = parse_fasta(all_sq); order = parse_stoichiometry(stoich); segs,pos=[],0
        for ch,cnt in order:
            base = cd.get(ch)
            if base is None: return [(0,len(seq))]
            for _ in range(cnt): segs.append((pos,pos+len(base))); pos+=len(base)
        return segs if pos==len(seq) else [(0,len(seq))]
    except: return [(0,len(seq))]

def build_segments_map(df):
    seg_map,stoich_map = {},{}
    for _,r in df.iterrows():
        tid = r["target_id"]; seg_map[tid] = get_chain_segments(r)
        raw_s = r.get("stoichiometry",""); stoich_map[tid] = "" if pd.isna(raw_s) else str(raw_s)
    return seg_map, stoich_map

def process_labels(labels_df):
    coords = {}
    prefixes = labels_df["ID"].str.rsplit("_", n=1).str[0]
    for prefix, grp in labels_df.groupby(prefixes):
        coords[prefix] = grp.sort_values("resid")[["x_1","y_1","z_1"]].values
    return coords

def _build_aligned_strings(query_seq, template_seq, alignment):
    q_segs,t_segs = alignment.aligned; aq,at,qi,ti = [],[],0,0
    for (qs,qe),(ts,te) in zip(q_segs,t_segs):
        while qi<qs: aq.append(query_seq[qi]); at.append("-"); qi+=1
        while ti<ts: aq.append("-"); at.append(template_seq[ti]); ti+=1
        for qp,tp in zip(range(qs,qe),range(ts,te)): aq.append(query_seq[qp]); at.append(template_seq[tp])
        qi,ti = qe,te
    while qi<len(query_seq): aq.append(query_seq[qi]); at.append("-"); qi+=1
    while ti<len(template_seq): aq.append("-"); at.append(template_seq[ti]); ti+=1
    return "".join(aq),"".join(at)

def find_similar_sequences_detailed(query_seq, train_seqs_df, train_coords_dict, top_n=30):
    results = []
    for _,row in train_seqs_df.iterrows():
        tid,tseq = row["target_id"],row["sequence"]
        if tid not in train_coords_dict: continue
        if abs(len(tseq)-len(query_seq))/max(len(tseq),len(query_seq)) > 0.3: continue
        aln = next(iter(_aligner.align(query_seq, tseq)))
        norm_s = aln.score/(2*min(len(query_seq),len(tseq)))
        identical = sum(1 for (qs,qe),(ts,te) in zip(*aln.aligned)
                        for qp,tp in zip(range(qs,qe),range(ts,te)) if query_seq[qp]==tseq[tp])
        pct_id = 100*identical/len(query_seq)
        aq,at = _build_aligned_strings(query_seq,tseq,aln)
        results.append((tid,tseq,norm_s,train_coords_dict[tid],pct_id,aq,at))
    results.sort(key=lambda x: x[2], reverse=True)
    return results[:top_n]

def adapt_template_to_query(query_seq, template_seq, template_coords):
    aln = next(iter(_aligner.align(query_seq, template_seq)))
    new_c = np.full((len(query_seq),3), np.nan)
    for (qs,qe),(ts,te) in zip(*aln.aligned):
        chunk = template_coords[ts:te]
        if len(chunk)==(qe-qs): new_c[qs:qe] = chunk
    for i in range(len(new_c)):
        if np.isnan(new_c[i,0]):
            pv = next((j for j in range(i-1,-1,-1) if not np.isnan(new_c[j,0])),-1)
            nv = next((j for j in range(i+1,len(new_c)) if not np.isnan(new_c[j,0])),-1)
            if pv>=0 and nv>=0: w=(i-pv)/(nv-pv); new_c[i]=(1-w)*new_c[pv]+w*new_c[nv]
            elif pv>=0: new_c[i]=new_c[pv]+[3,0,0]
            elif nv>=0: new_c[i]=new_c[nv]+[3,0,0]
            else: new_c[i]=[i*3,0,0]
    return np.nan_to_num(new_c)

def adaptive_rna_constraints(coords, target_id, segments_map, confidence=1.0, passes=2):
    X = coords.copy(); segments = segments_map.get(target_id, [(0,len(X))])
    strength = max(0.75*(1.0-min(confidence,0.97)), 0.02)
    for _ in range(passes):
        for s,e in segments:
            C = X[s:e]; L = e-s
            if L < 3: continue
            d = C[1:]-C[:-1]; dist = np.linalg.norm(d,axis=1)+1e-6
            adj = d*((5.95-dist)/dist)[:,None]*(0.22*strength)
            C[:-1]-=adj; C[1:]+=adj
            d2 = C[2:]-C[:-2]; d2n = np.linalg.norm(d2,axis=1)+1e-6
            adj2 = d2*((10.2-d2n)/d2n)[:,None]*(0.10*strength)
            C[:-2]-=adj2; C[2:]+=adj2
            C[1:-1]+=(0.06*strength)*(0.5*(C[:-2]+C[2:])-C[1:-1])
            if L >= 25:
                idx = np.linspace(0,L-1,min(L,160)).astype(int) if L>220 else np.arange(L)
                P = C[idx]; diff = P[:,None,:]-P[None,:,:]
                dm = np.linalg.norm(diff,axis=2)+1e-6; sep = np.abs(idx[:,None]-idx[None,:])
                mask = (sep>2)&(dm<3.2)
                if np.any(mask):
                    vec = (diff*((3.2-dm)/dm)[:,:,None]*mask[:,:,None]).sum(axis=1)
                    C[idx]+=(0.015*strength)*vec
            X[s:e] = C
    return X

def _rotmat(axis, ang):
    a = np.asarray(axis,float); a/=np.linalg.norm(a)+1e-12
    x,y,z=a; c,s=np.cos(ang),np.sin(ang); CC=1-c
    return np.array([[c+x*x*CC,x*y*CC-z*s,x*z*CC+y*s],
                     [y*x*CC+z*s,c+y*y*CC,y*z*CC-x*s],
                     [z*x*CC-y*s,z*y*CC+x*s,c+z*z*CC]])

def apply_hinge(coords, seg, rng, deg=22):
    s,e=seg; L=e-s
    if L<30: return coords
    pivot=s+int(rng.integers(10,L-10))
    R=_rotmat(rng.normal(size=3),np.deg2rad(float(rng.uniform(-deg,deg))))
    X=coords.copy(); p0=X[pivot].copy(); X[pivot+1:e]=(X[pivot+1:e]-p0)@R.T+p0
    return X

def jitter_chains(coords, segs, rng, deg=12, trans=1.5):
    X=coords.copy(); gc_=X.mean(0,keepdims=True)
    for s,e in segs:
        R=_rotmat(rng.normal(size=3),np.deg2rad(float(rng.uniform(-deg,deg))))
        shift=rng.normal(size=3); shift=shift/(np.linalg.norm(shift)+1e-12)*float(rng.uniform(0,trans))
        c=X[s:e].mean(0,keepdims=True); X[s:e]=(X[s:e]-c)@R.T+c+shift
    X-=X.mean(0,keepdims=True)-gc_; return X

def smooth_wiggle(coords, segs, rng, amp=0.8):
    X=coords.copy()
    for s,e in segs:
        L=e-s
        if L<20: continue
        ctrl=np.linspace(0,L-1,6); disp=rng.normal(0,amp,(6,3)); t=np.arange(L)
        X[s:e]+=np.vstack([np.interp(t,ctrl,disp[:,k]) for k in range(3)]).T
    return X

def generate_rna_structure(sequence, seed=None):
    if seed is not None: np.random.seed(seed)
    n=len(sequence); coords=np.zeros((n,3))
    for i in range(n):
        ang=i*0.6; coords[i]=[10.0*np.cos(ang),10.0*np.sin(ang),i*2.5]
    return coords

def tbm_phase(test_df, train_seqs_df, train_coords_dict, segments_map):
    print(f"\n{'='*60}\nPHASE 1: Template-Based Modeling")
    print(f"  MIN_SIMILARITY={MIN_SIMILARITY}  |  MIN_PCT_IDENTITY={MIN_PERCENT_IDENTITY}\n{'='*60}")
    t0=time.time(); template_predictions,protenix_queue = {},{}
    for _,row in test_df.iterrows():
        tid=row["target_id"]; seq=row["sequence"]; segs=segments_map.get(tid,[(0,len(seq))])
        similar=find_similar_sequences_detailed(seq,train_seqs_df,train_coords_dict,top_n=30)
        preds=[]; used=set()
        for i,(tmpl_id,tmpl_seq,sim,tmpl_coords,pct_id,_,_) in enumerate(similar):
            if len(preds)>=N_SAMPLE: break
            if sim<MIN_SIMILARITY or pct_id<MIN_PERCENT_IDENTITY: break
            if tmpl_id in used: continue
            rng=np.random.default_rng((row.name*10000000000+i*10007)%(2**32))
            adapted=adapt_template_to_query(seq,tmpl_seq,tmpl_coords)
            slot=len(preds)
            if slot==0: X=adapted
            elif slot==1: X=adapted+rng.normal(0,max(0.01,(0.40-sim)*0.06),adapted.shape)
            elif slot==2:
                longest=max(segs,key=lambda se:se[1]-se[0]); X=apply_hinge(adapted,longest,rng)
            elif slot==3: X=jitter_chains(adapted,segs,rng)
            else: X=smooth_wiggle(adapted,segs,rng)
            refined=adaptive_rna_constraints(X,tid,segments_map,confidence=sim)
            preds.append(refined); used.add(tmpl_id)
        template_predictions[tid]=preds
        n_needed=N_SAMPLE-len(preds)
        if n_needed>0:
            protenix_queue[tid]=(n_needed,seq)
            print(f"  {tid} ({len(seq)} nt): {len(preds)} TBM -> need {n_needed} Protenix")
        else:
            print(f"  {tid} ({len(seq)} nt): all {N_SAMPLE} TBM ok")
    elapsed=time.time()-t0
    print(f"\nPhase 1 done in {elapsed:.1f}s | TBM:{len(test_df)-len(protenix_queue)} | Protenix:{len(protenix_queue)}")
    return template_predictions,protenix_queue


In [4]:
def main():
    test_csv,output_csv,code_dir,root_dir = resolve_paths()
    if not os.path.isdir(code_dir):
        raise FileNotFoundError(f"Missing PROTENIX_CODE_DIR: {code_dir}")
    os.environ["PROTENIX_ROOT_DIR"]=root_dir; sys.path.append(code_dir)
    ensure_required_files(root_dir); seed_everything(SEED)
    print(f"GPUs: {torch.cuda.device_count()}")
    test_df=pd.read_csv(test_csv).reset_index(drop=True)
    print(f"Test targets: {len(test_df)}")
    train_seqs=pd.read_csv(DEFAULT_TRAIN_CSV); val_seqs=pd.read_csv(DEFAULT_VAL_CSV)
    train_labels=pd.read_csv(DEFAULT_TRAIN_LBLS); val_labels=pd.read_csv(DEFAULT_VAL_LBLS)
    combined_seqs=pd.concat([train_seqs,val_seqs],ignore_index=True)
    combined_labels=pd.concat([train_labels,val_labels],ignore_index=True)
    train_coords=process_labels(combined_labels)
    del train_labels,val_labels,combined_labels; gc.collect()
    segments_map,_=build_segments_map(test_df)
    print(f"Template pool: {len(combined_seqs)} seqs, {len(train_coords)} structures")
    template_preds,protenix_queue=tbm_phase(test_df,combined_seqs,train_coords,segments_map)
    protenix_preds={}
    if protenix_queue and USE_PROTENIX:
        print(f"\n{'='*60}\nPHASE 2: Protenix for {len(protenix_queue)} targets\n{'='*60}")
        work_dir=Path("/kaggle/working"); work_dir.mkdir(parents=True,exist_ok=True)
        tasks=[]; chunk_info={}
        for target_id,(n_needed,full_seq) in protenix_queue.items():
            seq_len=len(full_seq)
            if seq_len<=MAX_SEQ_LEN:
                tasks.append({"target_id":target_id,"sequence":full_seq})
                chunk_info[target_id]=[{"name":target_id,"range":(0,seq_len)}]
                print(f"  {target_id} ({seq_len} nt): single pass")
            else:
                chunks=split_into_chunks(seq_len,MAX_SEQ_LEN,CHUNK_OVERLAP)
                print(f"  {target_id} ({seq_len} nt): {len(chunks)} chunks")
                chunk_info[target_id]=[]
                for ci,(cs,ce) in enumerate(chunks):
                    cn=f"{target_id}_chunk{ci}"
                    tasks.append({"target_id":cn,"sequence":full_seq[cs:ce]})
                    chunk_info[target_id].append({"name":cn,"range":(cs,ce)})
        tasks_df=pd.DataFrame(tasks)
        input_json_path=str(work_dir/"protenix_queue_input.json")
        build_input_json(tasks_df,input_json_path)
        from protenix.data.inference.infer_dataloader import InferenceDataset
        from runner.inference import InferenceRunner,update_gpu_compatible_configs,update_inference_configs
        configs=build_configs(input_json_path,str(work_dir/"outputs"),MODEL_NAME)
        configs=update_gpu_compatible_configs(configs)
        runner=InferenceRunner(configs); dataset=InferenceDataset(configs)
        raw_predictions={}
        def _extract_c1(pred,data,atom_array,chunk_seq_len,raw_coords):
            if "centre_atom_mask" in data["input_feature_dict"]:
                mask=(data["input_feature_dict"]["centre_atom_mask"]==1).to(raw_coords.device)
            elif "atom_to_tokatom_idx" in data["input_feature_dict"]:
                m11=(data["input_feature_dict"]["atom_to_tokatom_idx"]==11).to(raw_coords.device)
                m12=(data["input_feature_dict"]["atom_to_tokatom_idx"]==12).to(raw_coords.device)
                mask=m11 if abs(m11.sum()-chunk_seq_len)<abs(m12.sum()-chunk_seq_len) else m12
            else:
                mask=torch.zeros(raw_coords.shape[1],dtype=torch.bool,device=raw_coords.device)
            coords=raw_coords[:,mask,:].detach().cpu().numpy()
            if coords.shape[1]>1:
                diffs=np.linalg.norm(coords[0,1:]-coords[0,:-1],axis=-1)
                if np.all(diffs<1e-4): print("    WARNING: collapsed"); return None
            if coords.shape[1]!=chunk_seq_len:
                if coords.shape[1]==1 and chunk_seq_len>1: return None
                padded=np.zeros((coords.shape[0],chunk_seq_len,3),dtype=np.float32)
                ml=min(coords.shape[1],chunk_seq_len); padded[:,:ml,:]=coords[:,:ml,:]
                coords=padded
            return coords
        for i in tqdm(range(len(dataset)),desc="Protenix"):
            data,atom_array,err=dataset[i]
            sample_name=data.get("sample_name",f"sample_{i}")
            if err:
                print(f"  {sample_name}: error - {err}"); raw_predictions[sample_name]=None
                del data,atom_array,err; gc.collect(); torch.cuda.empty_cache(); gc.collect()
                continue
            target_id=sample_name.split("_chunk")[0] if "_chunk" in sample_name else sample_name
            n_needed=protenix_queue.get(target_id,(N_SAMPLE,""))[0]
            sub_seq_len=data["N_token"].item()
            try:
                new_cfg=update_inference_configs(configs,sub_seq_len)
                new_cfg.sample_diffusion.N_sample=n_needed
                runner.update_model_configs(new_cfg)
                pred=runner.predict(data); raw_coords=pred["coordinate"]
                coords=_extract_c1(pred,data,atom_array,sub_seq_len,raw_coords)
                raw_predictions[sample_name]=coords
                if coords is not None: print(f"\n  {sample_name}: {coords.shape[0]} ok")
                else: print(f"\n  {sample_name}: extraction failed")
            except Exception as exc:
                import traceback
                print(f"\n  {sample_name}: FAILED - {exc}"); traceback.print_exc()
                raw_predictions[sample_name]=None
            finally:
                try: del pred,data,atom_array,raw_coords
                except: pass
                gc.collect(); torch.cuda.empty_cache(); gc.collect()
        for target_id,(n_needed,full_seq) in protenix_queue.items():
            seq_len=len(full_seq); chunks=chunk_info.get(target_id,[])
            if not chunks: continue
            if len(chunks)==1:
                coords=raw_predictions.get(target_id); protenix_preds[target_id]=coords
                if coords is not None: print(f"  {target_id}: {coords.shape[0]} preds ok")
                else: print(f"  {target_id}: FAILED")
            else:
                per_sample={s:[] for s in range(n_needed)}; all_ok=True
                for cinfo in chunks:
                    ccoords=raw_predictions.get(cinfo["name"])
                    if ccoords is None: all_ok=False; break
                    for s_idx in range(n_needed):
                        si=s_idx if s_idx<ccoords.shape[0] else -1
                        per_sample[s_idx].append((ccoords[si],cinfo["range"]))
                if not all_ok:
                    print(f"  {target_id}: chunked incomplete -> fallback")
                    protenix_preds[target_id]=None; continue
                stitched=[]
                for s_idx in range(n_needed):
                    items=per_sample[s_idx]
                    fc=stitch_chunk_coords([c for c,_ in items],[r for _,r in items],seq_len)
                    stitched.append(fc)
                result=np.stack(stitched,axis=0); protenix_preds[target_id]=result
                print(f"  {target_id}: {result.shape[0]} stitched ok")
    print(f"\n{'='*60}\nPHASE 3: Combine\n{'='*60}")
    all_rows=[]
    for _,row in test_df.iterrows():
        tid=row["target_id"]; seq=row["sequence"]
        combined=list(template_preds.get(tid,[]))
        ptx=protenix_preds.get(tid)
        if ptx is not None and ptx.ndim==3:
            for j in range(ptx.shape[0]):
                if len(combined)>=N_SAMPLE: break
                combined.append(ptx[j])
        while len(combined)<N_SAMPLE:
            seed_val=row.name*1000000+len(combined)*1000
            dn=generate_rna_structure(seq,seed=seed_val)
            combined.append(adaptive_rna_constraints(dn,tid,segments_map,confidence=0.2))
        stacked=np.stack(combined[:N_SAMPLE],axis=0)
        all_rows.extend(coords_to_rows(tid,seq,stacked))
    sub=pd.DataFrame(all_rows)
    cols=["ID","resname","resid"]+[f"{c}_{i}" for i in range(1,N_SAMPLE+1) for c in ["x","y","z"]]
    cc=[c for c in cols if c.startswith(("x_","y_","z_"))]
    sub[cc]=sub[cc].clip(-999.999,9999.999)
    sub[cols].to_csv(output_csv,index=False)
    print(f"\nDone | {len(sub):,} rows -> {output_csv}")


main()

GPUs: 2
Test targets: 28
Template pool: 5744 seqs, 5744 structures

PHASE 1: Template-Based Modeling
  MIN_SIMILARITY=0.0  |  MIN_PCT_IDENTITY=50.0
  8ZNQ (30 nt): 2 TBM -> need 3 Protenix
  9IWF (69 nt): all 5 TBM ok
  9JGM (210 nt): all 5 TBM ok
  9MME (4640 nt): 4 TBM -> need 1 Protenix
  9J09 (214 nt): 1 TBM -> need 4 Protenix
  9E9Q (101 nt): 4 TBM -> need 1 Protenix
  9CFN (59 nt): 2 TBM -> need 3 Protenix
  9OBM (73 nt): 3 TBM -> need 2 Protenix
  9G4P (68 nt): 4 TBM -> need 1 Protenix
  9G4Q (104 nt): 2 TBM -> need 3 Protenix
  9G4R (47 nt): 1 TBM -> need 4 Protenix
  9RVP (34 nt): 4 TBM -> need 1 Protenix
  9JFS (246 nt): 1 TBM -> need 4 Protenix
  9LEC (378 nt): 2 TBM -> need 3 Protenix
  9LEL (476 nt): 2 TBM -> need 3 Protenix
  9I9W (28 nt): all 5 TBM ok
  9HRO (35 nt): all 5 TBM ok
  9QZJ (19 nt): all 5 TBM ok
  9JFO (195 nt): all 5 TBM ok
  9OD4 (23 nt): all 5 TBM ok
  9WHV (80 nt): all 5 TBM ok
  9E74 (255 nt): all 5 TBM ok
  9E75 (165 nt): all 5 TBM ok
  9G4J (334 nt): 

2026-03-23 10:54:25,181 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:557] INFO runner.inference: Enforcing FP32 and torch kernels for compatibility with detected GPU (Compute Capability 7.x).
2026-03-23 10:54:25,182 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Distributed environment: world size: 1, global rank: 0, local rank: 0
2026-03-23 10:54:25,182 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:98] INFO root: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
2026-03-23 10:54:25,428 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:127] INFO root: Finished environment initialization.


train scheduler 16.0
inference scheduler 16.0
Diffusion Module has 16.0


2026-03-23 10:55:41,150 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Loading from /kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/checkpoint/protenix_base_20250630_v1.0.0.pt, strict: True
2026-03-23 10:55:50,728 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Sampled key: module.input_embedder.atom_attention_encoder.linear_no_bias_ref_pos.weight
2026-03-23 10:55:50,870 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] INFO runner.inference: Finish loading checkpoint.
2026-03-23 10:55:50,881 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/runner/inference.py:246] 


  8ZNQ: 3 ok


...


  9ZCC_chunk2: 3 ok


Protenix:  97%|█████████▋| 28/29 [2:06:08<06:09, 369.83s/it]2026-03-23 13:01:59,484 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/protenix/data/inference/infer_dataloader.py:281] INFO protenix.data.inference.infer_dataloader: Featurizing 9ZCC_chunk3...
2026-03-23 13:01:59,854 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/protenix/data/constraint/constraint_featurizer.py:392] INFO protenix.data.constraint.constraint_featurizer: Loaded constraint feature: #atom contact:0 #contact:0 #pocket:0
2026-03-23 13:02:00,381 [/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1/protenix/data/template/template_featurizer.py:668] INFO protenix.data.template.template_featurizer: Calling InferenceTemplateFeaturizer.make_template_feature



  9ZCC_chunk3: 3 ok


Protenix: 100%|██████████| 29/29 [2:08:58<00:00, 266.84s/it]


  8ZNQ: 3 preds ok
  9MME: 1 stitched ok
  9J09: 4 preds ok
  9E9Q: 1 preds ok
  9CFN: 3 preds ok
  9OBM: 2 preds ok
  9G4P: 1 preds ok
  9G4Q: 3 preds ok
  9G4R: 4 preds ok
  9RVP: 1 preds ok
  9JFS: 4 preds ok
  9LEC: 3 preds ok
  9LEL: 3 preds ok
  9EBP: 4 preds ok
  9ZCC: 3 stitched ok

PHASE 3: Combine

Done | 9,762 rows -> /kaggle/working/submission.csv
